# DeFi Data Pipeline — FinTech 590
Fetches top Uniswap V3 pools, verifies contracts on Etherscan, and saves results.

## 0. Install Dependencies

In [ ]:
import subprocess, sys

def ensure(*packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"[setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure("requests", "pandas", "python-dotenv", "web3")
print("All dependencies ready.")

## 1. Config & API Key

In [ ]:
import os, json, time, pathlib
import requests
import pandas as pd
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()
ETHERSCAN_API_KEY = os.getenv("ETHERSCAN_API_KEY")

if not ETHERSCAN_API_KEY:
    ETHERSCAN_API_KEY = input("Etherscan API key not found in .env — please paste it now: ").strip()
    if not ETHERSCAN_API_KEY:
        raise ValueError("Etherscan API key is required.")
    with open(".env", "a") as f:
        f.write(f"\nETHERSCAN_API_KEY={ETHERSCAN_API_KEY}\n")
    print("[setup] Key saved to .env for future runs.")
else:
    print("Etherscan API key loaded from .env.")

DEFILLAMA_POOLS  = "https://yields.llama.fi/pools"
ETHERSCAN_API    = "https://api.etherscan.io/api"
RATE_LIMIT_DELAY = 0.21   # 5 req/s free tier
MIN_TVL_USD      = 1_000_000
TOP_N            = 20

# Uniswap V3 factory constants (Ethereum mainnet)
UNISWAP_V3_FACTORY   = "0x1F98431c8aD98523631AE4a59f267346ea31F984"
POOL_INIT_CODE_HASH  = "e34f199b19b2b4f47f68442619d555527d244f78a3297ea89325f843f87b8b54"

ABI_DIR  = pathlib.Path("pool_abis")
CSV_PATH = pathlib.Path("top_pools.csv")
ABI_DIR.mkdir(exist_ok=True)

print(f"Output: {CSV_PATH}  |  ABIs: {ABI_DIR}/")

## 2. Fetch Top Uniswap V3 Pools from DeFiLlama

> **Note:** The Graph's free hosted service was shut down in 2024.  
> This step uses [DeFiLlama's free Yields API](https://yields.llama.fi/docs) — no API key required.  
> DeFiLlama returns internal UUIDs rather than on-chain addresses, so we compute the real
> pool address from token addresses + fee tier using Uniswap V3's deterministic CREATE2 formula.

In [ ]:
def compute_pool_address(token_a: str, token_b: str, fee: int) -> str:
    """
    Derive the Uniswap V3 pool address from its two tokens and fee tier
    using the same CREATE2 formula the factory contract uses on-chain.
    """
    # token0 must be the lower address
    token0, token1 = sorted([token_a.lower(), token_b.lower()])
    # ABI-encode the pool key (each param padded to 32 bytes)
    encoded = (
        bytes.fromhex(token0[2:]).rjust(32, b'\x00') +
        bytes.fromhex(token1[2:]).rjust(32, b'\x00') +
        fee.to_bytes(32, 'big')
    )
    salt = Web3.keccak(encoded)
    data = (
        b'\xff' +
        bytes.fromhex(UNISWAP_V3_FACTORY[2:]) +
        salt +
        bytes.fromhex(POOL_INIT_CODE_HASH)
    )
    return Web3.to_checksum_address('0x' + Web3.keccak(data)[12:].hex())


def parse_fee_tier(pool_meta: str | None) -> int:
    """Convert poolMeta string like '0.05%' to Uniswap fee pips (500)."""
    try:
        return int(float(pool_meta.replace("%", "")) * 10_000)
    except (AttributeError, ValueError):
        return 0


print("Fetching pool data from DeFiLlama...")
resp = requests.get(DEFILLAMA_POOLS, timeout=30)
resp.raise_for_status()
all_pools = resp.json()["data"]
print(f"  Total pools across all protocols: {len(all_pools):,}")

# Filter: Uniswap V3 on Ethereum, TVL > $1M
uni_pools = [
    p for p in all_pools
    if p.get("project") == "uniswap-v3"
    and p.get("chain") == "Ethereum"
    and (p.get("tvlUsd") or 0) >= MIN_TVL_USD
]
uni_pools.sort(key=lambda x: x.get("tvlUsd", 0), reverse=True)
top_raw = uni_pools[:TOP_N]
print(f"  Uniswap V3 / Ethereum / TVL>$1M: {len(uni_pools)} found, keeping top {len(top_raw)}\n")

pools = []
for p in top_raw:
    underlying = p.get("underlyingTokens") or []
    fee        = parse_fee_tier(p.get("poolMeta"))
    symbol_parts = p.get("symbol", "-").split("-")

    # Compute real on-chain address if we have both token addresses
    if len(underlying) >= 2 and fee > 0:
        address = compute_pool_address(underlying[0], underlying[1], fee)
    else:
        address = p["pool"]  # fallback to DeFiLlama UUID

    pools.append({
        "address":    address,
        "token0":     symbol_parts[0] if len(symbol_parts) > 0 else "",
        "token1":     symbol_parts[1] if len(symbol_parts) > 1 else "",
        "fee_tier":   fee,
        "tvl_usd":    float(p.get("tvlUsd") or 0),
        "volume_usd": float(p.get("volumeUsd1d") or 0),
    })

for i, p in enumerate(pools, 1):
    print(f"  {i:>2}. {p['token0']}/{p['token1']} "
          f"(fee {p['fee_tier']/1e4:.2f}%)  "
          f"TVL=${p['tvl_usd']:>14,.0f}  "
          f"addr={p['address'][:10]}...")

## 3. Verify Each Pool Contract on Etherscan

In [ ]:
def etherscan_get_abi(address: str) -> tuple[bool, list | None]:
    """Returns (is_verified, abi_or_None). Handles rate-limit with retry."""
    params = {
        "module":  "contract",
        "action":  "getabi",
        "address": address,
        "apikey":  ETHERSCAN_API_KEY,
    }
    for attempt in range(3):
        r = requests.get(ETHERSCAN_API, params=params, timeout=15)
        r.raise_for_status()
        payload = r.json()
        if payload["status"] == "1":
            return True, json.loads(payload["result"])
        if "rate limit" in payload.get("result", "").lower():
            print(f"    [rate-limit] backing off 2 s ...")
            time.sleep(2)
            continue
        return False, None
    return False, None


for p in pools:
    addr = p["address"]
    print(f"  Checking {addr[:10]}...  ({p['token0']}/{p['token1']})", end="", flush=True)
    verified, abi = etherscan_get_abi(addr)
    p["etherscan_verified"] = verified

    if verified and abi:
        abi_path = ABI_DIR / f"{addr}.json"
        abi_path.write_text(json.dumps(abi, indent=2))
        print(f"  ✓ verified  (ABI saved → pool_abis/{addr}.json)")
    else:
        print(f"  ✗ not verified")

    time.sleep(RATE_LIMIT_DELAY)

## 4. Save Results

In [ ]:
df = pd.DataFrame(pools, columns=[
    "address", "token0", "token1", "fee_tier",
    "tvl_usd", "volume_usd", "etherscan_verified"
])
df.to_csv(CSV_PATH, index=False)

verified_count = df["etherscan_verified"].sum()
abi_files      = list(ABI_DIR.glob("*.json"))

print(f"top_pools.csv  → {len(df)} rows saved")
print(f"pool_abis/     → {len(abi_files)} ABI files ({verified_count} verified pools)")
print()
df[["token0", "token1", "fee_tier", "tvl_usd", "etherscan_verified"]]